## Setup



In [ ]:
import astropy.units as u
from astropy.coordinates import SkyCoord
from regions import PointSkyRegion

# %matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from astropy.time import Time
import json

In [ ]:
from IPython.display import display
from gammapy.data import DataStore
from gammapy.datasets import Datasets, SpectrumDataset, FluxPointsDataset
from gammapy.makers import (
    ReflectedRegionsBackgroundMaker,
    SafeMaskMaker,
    SpectrumDatasetMaker,
    WobbleRegionsFinder,
)
from gammapy.maps import Map, MapAxis, RegionGeom
from gammapy.modeling import Fit
from gammapy.modeling.models import (
    LogParabolaSpectralModel,
    SkyModel,
    create_crab_spectral_model,
)
from astropy.table import vstack
from gammapy.estimators import FluxPointsEstimator, LightCurveEstimator, FluxPoints

In [ ]:
import logging

# ignores Errors and warnings
logging.getLogger("gammapy").setLevel(logging.ERROR)
logging.getLogger("gammapy.irf").setLevel(logging.ERROR)

## Load data


In [ ]:
# Adjust this path to where you copied (and unpacked!) the tarball:
data_path = ""
data_store = DataStore.from_dir(data_path)
observations = data_store.get_observations(required_irf="point-like")

len(observations)

## Define the ON region


In [ ]:
source_name="Mrk501"
target_position = SkyCoord.from_name(source_name, frame="icrs", cache=True)
on_region = PointSkyRegion(target_position)
target_position

## Definitions for the data reduction chain


In [ ]:
# true and estimated energy axes
energy_axis =MapAxis.from_energy_bounds(0.3, 35, nbin=5, unit="TeV", name="energy",per_decade=True)
energy_axis_true = MapAxis.from_energy_bounds(0.3, 35, nbin=5, unit="TeV", name="energy_true",per_decade=True)

# geometry defining the ON region and SpectrumDataset based on it
geom = RegionGeom.create(region=on_region, axes=[energy_axis])
dataset_empty = SpectrumDataset.create(geom=geom, energy_axis_true=energy_axis_true)
energy_axis.edges


The `SpectrumDataset` is now based on a geometry consisting of a
single coordinate and an estimated energy axis. The
`SpectrumDatasetMaker` and `ReflectedRegionsBackgroundMaker` will
take care of producing ON and OFF with the proper sizes, automatically
adopting the $\theta$ values in `Observation.rad_max`.

As explained in the introduction, we use a `WobbleRegionsFinder`, to
determine the OFF positions. The parameter `n_off_positions` specifies
the number of OFF regions to be considered.




In [ ]:
dataset_maker = SpectrumDatasetMaker(containment_correction=False, use_region_center=True, selection=["counts", "exposure", "edisp"])
safe_mask_maker = SafeMaskMaker(methods=["aeff-max"], aeff_percent=10)

# WobbleRegionsFinder, 3 off regions
region_finder = WobbleRegionsFinder(n_off_regions=3)
bkg_maker = ReflectedRegionsBackgroundMaker(region_finder=region_finder)

# use the energy threshold specified in the DL3 files
safe_mask_masker = SafeMaskMaker(methods=["aeff-default"])

## Quality Cuts

In [ ]:
# Histogramm of L3 values to check the range of values between 80-100Hz
import matplotlib.pyplot as plt
import numpy as np
from gammapy.data import DataStore

# 1. Daten laden
data_store = DataStore.from_dir(data_path)
tab = data_store.obs_table

l3 = np.array(tab["L3RATE"])

# Anzahl der hinzugewonnenen Runs zwischen 80 und 100 Hz dynamisch berechnen
n_gain = np.sum((l3 >= 80) & (l3 < 100))

# 2. Histogramm (0 bis 150 Hz, 2-Hz-Schritte)
plt.figure(figsize=(8, 5))

plt.hist(
    l3,
    bins=np.arange(0, 152, 2),
    color="steelblue",
    edgecolor="black",
    linewidth=0.5,
)

# Schnitte bei 80 und 100 Hz
plt.axvline(
    100, color="red", linestyle="--", linewidth=1.5, label=" Cut (100 Hz)"
)
plt.axvline(
    80,
    color="darkorange",
    linestyle="--",
    linewidth=1.5,
    label=f" Cut (80 Hz: +{n_gain} Runs)",
)

# Achsen & Layout
plt.xlim(0, 150)
plt.xlabel("L3 Trigger Rate [Hz]")
plt.ylabel("Anzahl Runs")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig("l3_distribution_simple.pdf")
plt.show()

In [ ]:
from gammapy.data import DataStore, Observation, Observations
from astropy.table import Table

def irf_integrety(observation: Observation, irfs: list[str] | str = None) -> bool:
    """Check IRFs for basic integrity like nans or zeros."""

    # resolve strings
    if irfs is None:
        irfs = observation.available_hdus
    elif irfs == "full-enclosure":
        irfs = ["events", "gti", "aeff", "edisp", "psf", "bkg"]
    elif irfs == "point-like":
        irfs = ["events", "gti", "aeff", "edisp"]

   # Prüft auf NaNs sowie borken IRFs (min == max)
    def is_broken(a: np.ndarray) -> bool:
        if a is None or a.size == 0:
            return True
        if np.isnan(a).any():
            return True
        if np.min(a) == np.max(a):  # fängt auch reine Nullen ab
            return True
        return False

    # basic integrety check for irf data
    try:
        if "events" in irfs:
            # ensure not empty
            event_list = observation.events
            if len(event_list.table) == 0:
                print("events")
                return False

        if "gti" in irfs:
            # ensure not empty
            gti = observation.gti
            if len(gti.table) == 0:
                print("gti")
                return False

        if "aeff" in irfs:
            if is_broken(observation.aeff.data):
                print("aeff")
                return False

        if "edisp" in irfs:
            if is_broken(observation.edisp.data):
                print("edisp")
                return False

        if "psf" in irfs:
            if is_broken(observation.psf.data):
                print("psf")
                return False

        if "rad_max" in irfs:
            if is_broken(observation.rad_max.data):
                print("rad_max")
                return False

        if "bkg" in irfs:
            if is_broken(observation.bkg.data):
                print("bkg")
                return False

    except Exception:
        return False

    return True


def data_selection(
    data_path,
    irfs,
    time_periods=None,
    GTI_min=5,
    TRANS_min=0.7,
    CLOUD_max=40,
    ZENITH_min=5,
    ZENITH_max=62,
    l3_min=80,
    l3_max=1000,
    DEADC_min=0.2,
    DEADC_max=1,
    AZ_min=0,
    AZ_max=360,
) -> Observations:
    """Apply a quality selection on the data and calculate the bakground in the on-region.

    Parameters:

    data : string
        Path to the DL3 files
    time_periods         :
        die zeit von der mc periode z.B. Time(['2023-03-10T11:00:00', '2024-02-09T11:00:00'], format='isot', scale='utc')
    analysis :
            name von der analyse um die daten zu verwalten, nur um zeug zu speichern
    ich hoffe die anderen parameter sind relativ verständlich

    das müsste die selektion sein:
    cuts = {'TRANS_min' :   0.7, 'brightness_max'   :   70, 'corr_min'  :   20, 'CLOUD_max' :   25, 'L3_min'    :   140, 'id_min'   :   45}
    """

    data_store = DataStore.from_dir(data_path)

    alt_max = 90 - ZENITH_min
    alt_min = 90 - ZENITH_max

    tab = data_store.obs_table

    # Flag observations passing cuts
    print("Percent removed by selection:")
    GTI_mask = ((tab["TSTOP"] - tab["TSTART"]) / 60) > GTI_min
    print(f"Observations Time: {(np.sum(~GTI_mask) / len(tab) * 100):.2f}")

    transmission_mask = (tab["TRANS9"] >= TRANS_min) | (
        (tab["TRANS9"] == -1) & (tab["CLOUD"] < CLOUD_max)
    )
    print(f"Transmission: {(np.sum(~transmission_mask) / len(tab) * 100):.2f}")

    zenith_mask = (tab["ALT_PNT"] >= alt_min) & (tab["ALT_PNT"] <= alt_max)
    print(f"Zenith Range: {(np.sum(~zenith_mask) / len(tab) * 100):.2f}")

    l3_mask = (tab["L3RATE"] >= l3_min) & (tab["L3RATE"] <= l3_max)
    print(f"L3 Trigger Rate: {(np.sum(~l3_mask) / len(tab) * 100):.2f}")

    DEADC_mask = (tab["DEADC"] >= DEADC_min) & (tab["DEADC"] <= DEADC_max)
    print(f"Deadtime: {(np.sum(~DEADC_mask) / len(tab) * 100):.2f}")

    AZ_mask = (tab["AZ_PNT"] >= AZ_min) & (tab["AZ_PNT"] <= AZ_max)
    print(f"Azimuth Range: {(np.sum(~AZ_mask) / len(tab) * 100):.2f}")

    mask = GTI_mask & transmission_mask & zenith_mask & l3_mask & DEADC_mask & AZ_mask

    ids = tab["OBS_ID"][mask]

    observations = data_store.get_observations(required_irf=irfs, obs_id=ids)

    if time_periods is not None:
        observations = observations.select_time(time_periods)

    # remove observations with edisp or aeff constant zero

    obs_before = len(observations)

    observations = Observations([o for o in observations if irf_integrety(o, irfs)])

    print(f"Integrety Check Passed: {((len(observations) / obs_before) * 100):.2f}%")

    return observations


In [ ]:
irfs = ["aeff", "edisp"]

# Zeitfenster 
time_periods = Time(["2020-01-01T00:00:00", "2025-12-01T00:00:00"], scale="utc")

# Funktionsaufruf
observation_list = data_selection(
    data_path=data_path,
    irfs=irfs,
    time_periods=time_periods,
    GTI_min=5,          # 5 Minuten = 300 Sekunden
    ZENITH_min=5,
    ZENITH_max=62,
    TRANS_min=0.7,
    CLOUD_max=40,
    DEADC_min=0.2,
    l3_min=80,
    l3_max=1000,
    # allowed_pointings=[...] 
)
print(len(observation_list))

In [ ]:
# IRF-Check auf ALLE autoMAGIC-Runs (gleiche Basis wie die anderen Prozentwerte)
data_store = DataStore.from_dir(data_path)
tab = data_store.obs_table
n_all = len(tab)

# Runs ohne Aeff/Edisp-HDU werden hier übersprungen -> zählen als "missing"
obs_all = data_store.get_observations(obs_id=tab["OBS_ID"], required_irf=irfs, skip_missing=True)
n_missing = n_all - len(obs_all)

# Runs mit vorhandenen, aber kaputten IRFs (NaN oder konstant)
broken_ids = [o.obs_id for o in obs_all if not irf_integrety(o, irfs)]
n_broken = len(broken_ids)

print(f"Runs gesamt (N_auto): {n_all}")
print(f"IRF missing:          {100 * n_missing / n_all:.2f} %  ({n_missing} Runs)")
print(f"IRF broken:           {100 * n_broken / n_all:.2f} %  ({n_broken} Runs)")
print(f"IRF gesamt entfernt:  {100 * (n_missing + n_broken) / n_all:.2f} %")
print("kaputte OBS_IDs:", broken_ids)

In [ ]:
from gammapy.utils.parallel import run_multiprocessing
from gammapy.datasets import Datasets


def reduce_observation(obs):
    dataset = dataset_maker.run(dataset_empty.copy(name=f"obs-{obs.obs_id}"), obs)
    dataset_on_off = bkg_maker.run(dataset=dataset, observation=obs)
    dataset_on_off = safe_mask_maker.run(dataset_on_off, obs)
    return dataset_on_off


inputs_for_multiprocessing = [(obs,) for obs in observation_list]

# Ausführung mit Gammapys Parallelisierung
print("Starte parallele Datenreduktion auf 4 Kernen...")
reduced_list = run_multiprocessing(
    func=reduce_observation,
    inputs=inputs_for_multiprocessing,
    backend="multiprocessing",
    pool_kwargs=dict(processes=4),
    task_name="Data Reduction" 
)

#  Liste wieder in das 'Datasets'-Objekt umwandeln
datasets_mrk = Datasets(reduced_list)
print("Fertig!")

In [ ]:
# Mrk501 Model
model_mrk501 = LogParabolaSpectralModel(
    alpha=2.4,
    amplitude="3.0e-12 cm-2 s-1 TeV-1",
    reference=1 * u.TeV,
    beta=0.4,
)
print(model_mrk501)
model_mrk = SkyModel(spectral_model = model_mrk501, name="Mrk501")

datasets_mrk.models = [model_mrk]

## Fit spectrum

joint fit from
  $300\,{\rm GeV}$ to $35\,{\rm TeV}$.




In [ ]:
# TEST OHNE STACKING. 
# define energy range for the fit
energy_range_mrk = [0.3, 35]*u.TeV
mask_fit = geom.energy_mask(energy_min=min(energy_range_mrk), energy_max=max(energy_range_mrk))
for dataset in datasets_mrk:
  dataset.mask_fit = mask_fit

fit = Fit()
result = fit.run(datasets=datasets_mrk)
print(result)
display(datasets_mrk.models.to_parameters_table())

## Compute the Spectral Flux Points



In [ ]:
fpe = FluxPointsEstimator(
    energy_edges=energy_axis.edges,
    source="Mrk501",
    selection_optional=["all"],
    n_sigma_ul=2,
    n_jobs=4,
)
flux_points = fpe.run(datasets=datasets_mrk)  # 'datasets_mrk' einsetzen

flux_points_dataset = FluxPointsDataset(data=flux_points, models=model_mrk)

## Spectrum

Let us compute the spectrum and compare the spectrum we obtained against a [previous measurement
by
MAGIC](https://ui.adsabs.harvard.edu/abs/2015JHEAp...5...30A/abstract)_.




In [ ]:
fig, ax_spectrum = plt.subplots()
flux_points_dataset.plot_spectrum(ax_spectrum)

plt.legend()
plt.grid()
plt.savefig("spectrum.pdf")
plt.show()
plt.clf()

In [ ]:
# Speichern der Werte vom globalen Fit für den Vergleich zwischen Bayesian Blocks später 
from pathlib import Path
from gammapy.modeling.models import Models

GLOBAL_FP_FILE    = Path("sed_global_flux_points.fits")
GLOBAL_MODEL_FILE = Path("sed_global_model.yaml")

flux_points_dataset.data.write(GLOBAL_FP_FILE, overwrite=True)
Models([model_mrk]).write(GLOBAL_MODEL_FILE, overwrite=True)

# Runs/Livetime für die Legende (Info nur zur Kontrolle)
n_runs_global  = len(datasets_mrk)
live_global_h  = sum(d.gti.time_sum.to_value("h") for d in datasets_mrk)
print(f"Global gespeichert: {n_runs_global} Runs")

## Light Curve

Apart from the spectrum, we are also interested in the light curve above 300 GeV.


### Run Wise Light Curve

In [ ]:
energy_range_mrk_lc = [0.3, 35]*u.TeV

# run wise
print(Time(min([o.tstart.mjd for o in observation_list]), format='mjd').iso)
time_intervals_rw = [Time([o.tstart, o.tstop]) for o in observation_list]
lc_maker_rw = LightCurveEstimator(
    energy_edges=energy_range_mrk_lc, 
    time_intervals=time_intervals_rw,
    source=source_name,
    reoptimize=False,
    n_sigma_ul=2,
    selection_optional="all",
    n_jobs=4
)
result_rw = lc_maker_rw.run(datasets_mrk)

fig, ax = plt.subplots()
fig.set_figwidth(15)
result_rw.plot(ax=ax, sed_type="flux", marker="o",alpha=0.4, label="Run-wise binning")
#ax.axhline(y=1.2e-10, linestyle='--', color='Grey', label="MAGIC Crab Reference (arXiv:1406.6892)")
ax.set_yscale('log')
plt.legend()
plt.tight_layout()
plt.savefig("ligh.pdf", bbox_inches="tight")
plt.show()
plt.clf()

### Night Wise Light Curve

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd

energy_range_mrk_lc = [0.3, 35]*u.TeV
# 1. Gamma-Daten 
day_of_first_observation = int(min([o.tstart.mjd for o in observation_list]))
day_of_last_observation = int(max([o.tstart.mjd for o in observation_list]))
n_time_bins = day_of_last_observation - day_of_first_observation + 3
edges_time_bins = day_of_first_observation + np.arange(n_time_bins)
time_intervals_nw = [Time([t-0.5, t+0.5], format="mjd") for t in edges_time_bins[:-1]]

print("Anzahl der Runs vor dem Binning:", len(observation_list))

lc_maker_nw = LightCurveEstimator(
    energy_edges=energy_range_mrk_lc,
    time_intervals=time_intervals_nw,
    source=source_name,
    reoptimize=False,
    n_sigma_ul=2,
    selection_optional="all",
    n_jobs=4
)
result_nw = lc_maker_nw.run(datasets_mrk)

In [ ]:
# OG Version 
import pandas as pd
import numpy as np
from gammapy.estimators import FluxPoints
import matplotlib.dates as mdates
flux_table = result_nw.to_table(sed_type="flux", format="lightcurve")

# Maske erstellen: is_ul == False und keine NaN im Fluss
mask_ul = flux_table["is_ul"][:, 0] == False
mask_nan = ~np.isnan(flux_table["flux"][:, 0])
mask_valid = mask_ul & mask_nan

flux_table_filtered = flux_table[mask_valid]

# Gefilterte Tabelle speichern (wichtig für LCCF und Bayesian Blocks)
flux_table_filtered.write("table_lc_filtered.fits", overwrite=True)

# Neues FluxPoints-Objekt für den Plot erzeugen (mit Beibehaltung Modells)
result_nw_filtered = FluxPoints.from_table(
    flux_table_filtered,
    sed_type="flux",
    format="lightcurve",
    reference_model=model_mrk.spectral_model,
)

print(f"Gültige MAGIC-Nächte nach Filterung: {len(flux_table_filtered)}")

# 2. Radio Daten (1. Band)
df_radio = pd.read_csv('Mrk501.csv')
df_7mm = df_radio[df_radio['Band [mm]'] == 7].copy()
t_radio_mjd = df_7mm['MJD'].values
t_radio_datetime = Time(t_radio_mjd, format='mjd').datetime
f_radio = df_7mm['Flux Density [Jy]'].values
e_radio = df_7mm['Error [Jy]'].values
#3. Radio Daten (2.Band)
df_14mm = df_radio[df_radio['Band [mm]'] == 14].copy()
t_radio_mjd_14mm = df_14mm['MJD'].values
t_radio_datetime_14mm = Time(t_radio_mjd_14mm, format='mjd').datetime
f_radio_14mm = df_14mm['Flux Density [Jy]'].values
e_radio_14mm = df_14mm['Error [Jy]'].values

n_radio_7mm = len(df_7mm)
n_radio_14mm = len(df_14mm)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
# Farben in iceblue
color_gamma = "#03045E"
ecolor_gamma = "#7294D4"
color_radio = "#0077B6"      
ecolor_radio = "#90E0EF"

# Oberer Plot: Gamma Lightcurve (MAGIC)
# Panel 1: MAGIC Gamma
result_nw_filtered.plot(
    ax=ax1, sed_type="flux", marker="o", alpha=0.8, label="MAGIC ($0.3$–$35$ TeV)"
)
ax1.set_yscale('log')
ax1.set_ylabel(r'$\Phi_{0.3-35 \text{ TeV}} \text{ [cm}^{-2}\text{s}^{-1}\text{]}$')
ax1.grid(True, alpha=0.3)
ax1.legend()

# --- Panel 2: Radio (7 mm) ---
ax2.errorbar(
    t_radio_datetime,
    f_radio,
    yerr=e_radio,
    fmt="o",
    color="tab:red",
    ecolor="lightcoral",
    elinewidth=1,
    capsize=2,
    label="Effelsberg (Radio, 7 mm)",
)
ax2.set_ylabel(r"$S_{\mathrm{7\,mm}}$ [Jy]")
ax2.grid(True, alpha=0.3)
ax2.legend(loc="upper right")

# --- Panel 3: Radio (14 mm) ---
ax3.errorbar(
    t_radio_datetime_14mm,
    f_radio_14mm,
    yerr=e_radio_14mm,
    fmt="o",
    color="tab:red",
    ecolor="lightcoral",
    elinewidth=1,
    capsize=2,
    label="Effelsberg (Radio, 14 mm)",
)
ax3.set_ylabel(r"$S_{\mathrm{14\,mm}}$ [Jy]")
ax3.set_xlabel("Time")
ax3.grid(True, alpha=0.3)
ax3.legend(loc="upper right")

# Zeitachsen-Formatierung am untersten Subplot
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
fig.autofmt_xdate()

plt.tight_layout()
plt.subplots_adjust(hspace=0.08)
plt.savefig("images/mrk501_lightcurve_all_bands.pdf", format="pdf")
plt.show()

def print_values(name, t):
    t_sorted = np.sort(t)  # Stellt sicher, dass chronologisch vorwärts gerechnet wird
    dt = np.diff(t_sorted)
    print(f"\n--- {name} ---")
    print(f"Punkte (N):     {len(t)}")
    print(f"Median dt:      {np.median(dt):.2f} Tage")
    print(f"Mittelwert dt:  {np.mean(dt):.2f} ± {np.std(dt):.2f} Tage")
    print(f"Min dt:         {np.min(dt):.2f} Tage")
    print(f"Max dt (Lücke): {np.max(dt):.2f} Tage")
    print(f"Nyquist-Limit:  {2.0 * np.median(dt):.2f} Tage")


t_magic = (flux_table["time_min"] + flux_table["time_max"]) / 2.0
if hasattr(t_magic, "value"):
    t_magic = t_magic.value

# Direkt ausgeben
print_values("MAGIC", t_magic)
print_values("Effelsberg 7 mm", t_radio_mjd)
print_values("Effelsberg 14 mm", t_radio_mjd_14mm)
print(model_mrk.spectral_model)

In [ ]:
#Max/Min Werte aus fits Tabelle holen

# Bayesian - Blocks Algorithmus

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.table import Table
from pathlib import Path
from astropy.time import Time
from astropy.stats import bayesian_blocks
import matplotlib.dates as mdates

flux_table = Table.read("table_lc_filtered.fits")
time_mjd = (flux_table["time_min"] + flux_table["time_max"]) / 2.0
flux = flux_table["flux"].quantity.value.flatten()
flux_err = flux_table["flux_err"].quantity.value.flatten()

bayesian_edges = bayesian_blocks(
    t=time_mjd, x=flux, sigma=flux_err, fitness="measures", p0=0.01
)

n_blocks = len(bayesian_edges) - 1
print(f"Anzahl gefundener Bayesian Blocks: {n_blocks}")

# Zeitskala nicht kontinuierlich -- Fix
t_start = flux_table["time_min"].min()
t_stop = flux_table["time_max"].max()

if hasattr(t_start, 'mjd'):
    t_start = t_start.mjd
    t_stop = t_stop.mjd

bayesian_edges[0] = t_start
bayesian_edges[-1] = t_stop

fig, ax = plt.subplots(figsize=(12, 6))
time_datetime = Time(time_mjd, format="mjd").datetime

# Originale Daten als Punkte
ax.errorbar(
    time_datetime, flux, yerr=flux_err, 
    fmt="o", label="MAGIC Data (1-Day)", alpha=0.6, color="tab:blue"
)

# Statistik pro Block (für Low/High-State-Auswahl)
block_stats = []

# Bayesian Blocks als Treppenstufen
for i in range(len(bayesian_edges) - 1):
    t_start, t_stop = bayesian_edges[i], bayesian_edges[i+1]
    mask_block = (time_mjd >= t_start) & (time_mjd <= t_stop)
    
    if np.sum(mask_block) > 0:
        weights = 1.0 / (flux_err[mask_block] ** 2)
        block_flux = np.average(flux[mask_block], weights=weights)
        block_stats.append(dict(
            block=i + 1, t_start=float(t_start), t_stop=float(t_stop),
            n_nights=int(np.sum(mask_block)), flux=float(block_flux),
        ))
        dt_start, dt_stop = Time(t_start, format="mjd").datetime, Time(t_stop, format="mjd").datetime
        
        # Horizontale Linie
        ax.hlines(
            y=block_flux, xmin=dt_start, xmax=dt_stop, 
            color="tab:orange", linewidth=3, 
            label="Bayesian Blocks" if i == 0 else ""
        )
        
        # Vertikale Linie 
        if i < len(bayesian_edges) - 2:
            next_mask = (time_mjd >= bayesian_edges[i+1]) & (time_mjd <= bayesian_edges[i+2])
            if np.sum(next_mask) > 0:
                next_weights = 1.0 / (flux_err[next_mask] ** 2)
                next_flux = np.average(flux[next_mask], weights=next_weights)
                ax.vlines(
                    x=dt_stop, ymin=min(block_flux, next_flux), ymax=max(block_flux, next_flux), 
                    color="tab:orange", linewidth=3
                )

ax.set_yscale('log')
ax.set_ylabel(r"$\Phi_{0.3-35 \text{ TeV}} \text{ [cm}^{-2}\text{s}^{-1}\text{]}$", fontsize=12)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
fig.autofmt_xdate() 
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("mrk501_bayesian_lightcurve.pdf")
plt.show()
plt.close()

# Low- und High-State-Blöcke für den Vergleich mit der globalen SED
MIN_NIGHTS = 3     # absolute Mindestanzahl Nächte pro Block (kein Median mehr)
N_TOP = 3

df_lc_blocks = pd.DataFrame(block_stats)
df_lc_blocks["start"] = Time(df_lc_blocks["t_start"].values, format="mjd").iso
df_lc_blocks["stop"]  = Time(df_lc_blocks["t_stop"].values,  format="mjd").iso

# Low- and High-State block selection for global SED comparison
# Logic: Filter blocks with n_nights >= median, then rank by flux
# Implementation adapted from a code suggestion by Claude (verified afterwards)
print(df_lc_blocks.sort_values("flux", ascending=False)[
    ["block", "start", "stop", "n_nights", "flux"]
].to_string(index=False, formatters={
    "start": lambda s: s[:10], "stop": lambda s: s[:10], "flux": "{:.2e}".format}))

cand = df_lc_blocks[df_lc_blocks["n_nights"] >= MIN_NIGHTS]
high_blocks = cand.nlargest(N_TOP, "flux")
low_blocks = cand[~cand["block"].isin(high_blocks["block"])].nsmallest(N_TOP, "flux")
low_block_ids  = low_blocks["block"].astype(int).tolist()
high_block_ids = high_blocks["block"].astype(int).tolist()
print("High:", high_block_ids, "  Low:", low_block_ids)

In [ ]:
# ==============================================================================
# SED per Bayesian Block (Top 3 High- & Low-State Blocks)
# ==============================================================================
from pathlib import Path
from astropy.time import Time
import astropy.units as u
from gammapy.datasets import Datasets
from gammapy.estimators import FluxPointsEstimator
from gammapy.modeling import Fit
from gammapy.modeling.models import LogParabolaSpectralModel, SkyModel
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

datasets_src = datasets_mrk
MIN_RUNS = 4
output_dir = Path("sed_bayesian_blocks")
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Assign runs to blocks via midpoint
t_mid = np.array([0.5 * (d.gti.time_start.mjd.min() + d.gti.time_stop.mjd.max()) for d in datasets_src])
block_idx = np.clip(np.searchsorted(bayesian_edges, t_mid, side="right") - 1, 0, len(bayesian_edges) - 2)

selected_blocks = pd.concat([
    high_blocks.assign(state="High State"),
    low_blocks.assign(state="Low State")
]).reset_index(drop=True)

fpe = FluxPointsEstimator(
    energy_edges=energy_axis.edges, source="Mrk501", selection_optional=["all"], n_sigma_ul=2, n_jobs=4
)

summary = []

# 2. SED Fitting & Parameter Extraction
for _, row in selected_blocks.iterrows():
    b_id = int(row["block"])
    state = row["state"]
    t0, t1 = bayesian_edges[b_id - 1], bayesian_edges[b_id]
    d_start, d_stop = Time([t0, t1], format="mjd").to_datetime()

    # Select block runs
    members = [d for d, b in zip(datasets_src, block_idx) if b == (b_id - 1)]
    if len(members) < MIN_RUNS:
        continue

    ds_block = Datasets([d.copy(name=f"b{b_id:02d}-{d.name}") for d in members])
    spec = LogParabolaSpectralModel(
        alpha=2.75, beta=0.28, reference=1 * u.TeV, amplitude=1e-12 * u.Unit("cm-2 s-1 TeV-1")
    )
    ds_block.models = [SkyModel(spectral_model=spec, name="Mrk501")]

    # Fit & Error Handling
    try:
        fit_res = Fit().run(datasets=ds_block)
        if not fit_res.success:
            raise RuntimeError("Fit failed to converge")
        flux_points = fpe.run(datasets=ds_block)
        success = True
    except Exception as e:
        print(f"[{state}] Block {b_id:02d}: {e}")
        success = False

    # Collect results for table
    res_dict = {
        "block": b_id, "state": state,
        "date_start": d_start.strftime("%Y-%m-%d"), "date_stop": d_stop.strftime("%Y-%m-%d"),
        "t_start": t0, "t_stop": t1, "n_runs": len(members), "success": success
    }

    if success:
        res_dict.update({
            "livetime_h": sum(d.gti.time_sum.to_value("h") for d in ds_block),
            "alpha": spec.alpha.value, "alpha_err": spec.alpha.error,
            "beta": spec.beta.value, "beta_err": spec.beta.error,
            "amplitude": spec.amplitude.value, "amplitude_err": spec.amplitude.error
        })

        # Save SED products & quick plot
        prefix = f"sed_{state.lower().replace(' ', '_')}_block_{b_id:02d}"
        flux_points.write(output_dir / f"{prefix}_flux_points.fits", overwrite=True)
        ds_block.models.write(output_dir / f"{prefix}_model.yaml", overwrite=True)

        fig, ax = plt.subplots(figsize=(6, 4.5))
        flux_points.plot(ax=ax, sed_type="e2dnde", label=f"Block {b_id} ({state})")
        color = "crimson" if state == "High State" else "navy"
        spec.plot(ax=ax, energy_bounds=(0.3, 35) * u.TeV, sed_type="e2dnde", color=color, label="Fit")
        spec.plot_error(ax=ax, energy_bounds=(0.3, 35) * u.TeV, sed_type="e2dnde", facecolor=color, alpha=0.25)
        ax.set_ylim(1e-13, 2e-10)
        ax.grid(which="both", alpha=0.3)
        ax.legend(loc="lower left")
        plt.savefig(output_dir / f"{prefix}.pdf", bbox_inches="tight")
        plt.close(fig)

    summary.append(res_dict)

# 3. Export Summary Table
df_blocks = pd.DataFrame(summary).sort_values(by=["state", "t_start"])
df_blocks.to_csv(output_dir / "top_states_sed_summary.csv", index=False)

print("\n" + "=" * 65)
print(df_blocks[["block", "state", "date_start", "date_stop", "n_runs", "alpha", "beta"]].to_string(index=False))
print("=" * 65)

In [ ]:
import os
from pathlib import Path
import astropy.units as u
from gammapy.estimators import FluxPoints
from gammapy.modeling.models import Models
import matplotlib.pyplot as plt

DATA_DIR = Path("sed_bayesian_blocks")

# Globaler Fit
FILE_FP_GLOBAL = Path("sed_global_flux_points.fits")
FILE_MOD_GLOBAL = Path("sed_global_model.yaml")

# High State (Block 5)
FILE_FP_HIGH = DATA_DIR / "sed_high_state_block_05_flux_points.fits"
FILE_MOD_HIGH = DATA_DIR / "sed_high_state_block_05_model.yaml"

# Low State (Block 15)
FILE_FP_LOW = DATA_DIR / "sed_low_state_block_15_flux_points.fits"
FILE_MOD_LOW = DATA_DIR / "sed_low_state_block_15_model.yaml"

def plot_state(
    ax,
    fp_path,
    mod_path,
    color,
    err_color,
    label,
    energy_bounds,
    linestyle="-",
):
  if not fp_path.exists():
    print(f"Hinweis: FluxPoints-Datei nicht gefunden: {fp_path}")
    return

  # Datenpunkte plotten
  fp = FluxPoints.read(fp_path)
  fp.plot(ax=ax, sed_type="e2dnde", color=color, label=f"{label} (Data)")

  # Spektralmodell plotten
  if mod_path.exists():
    models = Models.read(mod_path)
    model = models["Mrk501"].spectral_model
    model.plot(
        ax=ax,
        energy_bounds=energy_bounds,
        sed_type="e2dnde",
        color=color,
        linestyle=linestyle,
        linewidth=1.8,
        label=rf"{label} ($\alpha={model.alpha.value:.2f}$)",
    )
    model.plot_error(
        ax=ax,
        energy_bounds=energy_bounds,
        sed_type="e2dnde",
        facecolor=err_color,
        alpha=0.25,
    )
  else:
    print(f"Hinweis: Model-Datei nicht gefunden: {mod_path}")



# Kombiniertes Spektrum zeichnen

fig, ax = plt.subplots(figsize=(8.0, 5.8))
energy_bounds = [0.3, 35] * u.TeV

# Farben definieren
COLOR_GLOBAL = "#2B2D42"  # Dunkles grau
COLOR_HIGH = "#D90429"  # Kräftiges Rot 
COLOR_HIGH_ERR = "#EF233C"
COLOR_LOW = "#0077B6"  # Blau 
COLOR_LOW_ERR = "#90E0EF"

# 1. Global State
plot_state(
    ax,
    FILE_FP_GLOBAL,
    FILE_MOD_GLOBAL,
    COLOR_GLOBAL,
    "gray",
    "Global",
    energy_bounds,
    linestyle=":",
)

# 2. High State (Block 05)
plot_state(
    ax,
    FILE_FP_HIGH,
    FILE_MOD_HIGH,
    COLOR_HIGH,
    COLOR_HIGH_ERR,
    "High State Fit Block 05",
    energy_bounds,
    linestyle="-",
)

# 3. Low State (Block 15
plot_state(
    ax,
    FILE_FP_LOW,
    FILE_MOD_LOW,
    COLOR_LOW,
    COLOR_LOW_ERR,
    "Low State Fit Block 15",
    energy_bounds,
    linestyle="-",
)

ax.set_ylim(5e-15, 5e-11)
ax.set_xlim(0.25, 40)
ax.set_ylabel(
    r"$E^2 \, \frac{\mathrm{d}N}{\mathrm{d}E} \; \left[ \mathrm{erg} \, \mathrm{cm}^{-2} \, \mathrm{s}^{-1} \right]$",
    fontsize=12,
)
ax.set_xlabel(r"$E \; [\mathrm{TeV}]$", fontsize=12)
ax.grid(True, which="both", linestyle="--", alpha=0.3)
ax.legend(loc="lower left", frameon=True, facecolor="white", framealpha=0.9)

plt.tight_layout()
output_file = DATA_DIR / "mrk501_combined_sed_states.pdf"
plt.savefig(output_file, dpi=300)
plt.savefig("mrk501_combined_sed_states.pdf", dpi=300)
plt.show()
print(f"Kombinierter Plot erfolgreich gespeichert unter: {output_file}")

## Algorithmus für Local Cross-Correlation Function (LCCF) von Max-Moerbeck et al. und Welsh für ungleich verteilte Lichtkurven

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time
from gammapy.estimators import FluxPoints
from astropy.table import Table

# Direkt aus der FITS-Datei laden
flux_table_filtered = Table.read("table_lc_filtered.fits")

# Zeit berechnen (MJD-Mitte des Intervalls)
t_gamma = (flux_table_filtered["time_min"].data + flux_table_filtered["time_max"].data) / 2.0
y_gamma = np.squeeze(flux_table_filtered["flux"].data) 
e_gamma = np.squeeze(flux_table_filtered["flux_err"].data)

# 2. Radio-Daten extrahieren
df_radio = pd.read_csv('Mrk501.csv')

df_7mm = df_radio[df_radio['Band [mm]'] == 7].copy()
t_radio = df_7mm['MJD'].values
f_radio = df_7mm['Flux Density [Jy]'].values
e_radio = df_7mm['Error [Jy]'].values

df_14mm = df_radio[df_radio['Band [mm]'] == 14].copy()
t_radio_14mm = df_14mm['MJD'].values
f_radio_14mm = df_14mm['Flux Density [Jy]'].values
e_radio_14mm = df_14mm['Error [Jy]'].values


def calculate_lccf(t1, y1, t2, y2, lag_bins, min_pairs=10):
    """
    Local Cross-Correlation Function (Welsh 1999).
    t1, y1: erste Lichtkurve (Gamma); t2, y2: zweite Lichtkurve (Radio).
    Lag tau = t2 - t1, also Radio - Gamma (tau > 0: Radio hinkt hinterher).
    Bins mit weniger als min_pairs gueltigen Paaren ergeben NaN.
    """
    dt = t2[:, None] - t1[None, :]                    # Matrix aller Zeitdifferenzen
    lags = (lag_bins[:-1] + lag_bins[1:]) / 2.0       # Bin-Mitten
    lccf_vals = []
    pair_counts = []

    for i in range(len(lag_bins) - 1):
        bin_mask = (dt >= lag_bins[i]) & (dt < lag_bins[i + 1])
        idx2, idx1 = np.where(bin_mask)               # Indizes der Paare in diesem Bin
        a_i = y1[idx1]                                # Gamma-Werte
        b_j = y2[idx2]                                # Radio-Werte

        # NaN-Werte herausfiltern
        valid = ~np.isnan(a_i) & ~np.isnan(b_j)
        a_i, b_j = a_i[valid], b_j[valid]
        m_valid = len(a_i)
        pair_counts.append(m_valid)                   # Anzahl gültiger Paare

        val = np.nan
        if m_valid >= min_pairs:
            mu_a, sigma_a = np.mean(a_i), np.std(a_i)
            mu_b, sigma_b = np.mean(b_j), np.std(b_j)
            if sigma_a > 0 and sigma_b > 0:
                val = np.mean((a_i - mu_a) * (b_j - mu_b)) / (sigma_a * sigma_b)
        lccf_vals.append(val)

    return lags, np.array(lccf_vals), np.array(pair_counts)


# 4. Berechnung für ein definiertes Binning (z. B. 10 Tage oder 30 Tage)
bin_width = 15.0
bins = np.arange(-500, 500 + bin_width, bin_width)

lags_7mm,  lccf_7mm,  pairs_7mm  = calculate_lccf(t_gamma, y_gamma, t_radio,f_radio, bins, min_pairs=10)
lags_14mm, lccf_14mm, pairs_14mm = calculate_lccf(t_gamma, y_gamma, t_radio_14mm, f_radio_14mm, bins, min_pairs=10)


# 5. Plot-Funktion für 2-Panel LCCF + Paarverteilung
def plot_lccf_with_pairs(lags, lccf_values, pair_counts, bin_w, title, filename):
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(9, 6), sharex=True, 
        gridspec_kw={'height_ratios': [2.5, 1], 'hspace': 0.08}
    )
    
    # Oberes Panel: LCCF
    ax1.scatter(lags, lccf_values, color='royalblue', s=35, zorder=3, label="LCCF")
    #ax1.plot(lags, lccf_values, color='royalblue', alpha=0.5, lw=1.2, zorder=2)
    ax1.axhline(0, color='black', lw=1, ls='--', alpha=0.7)
    ax1.set_ylabel("LCCF Value $r$", fontsize=11)
    ax1.set_ylim(-1, 1)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper right')
    
    # Unteres Panel: Anzahl der Paare pro Bin
    ax2.bar(lags, pair_counts, width=bin_w * 0.85, color='gray', alpha=0.7, edgecolor='black', lw=0.5)
    ax2.set_xlabel(r"Time Lag $\tau$ [Days] (Radio $-$ $\gamma$)", fontsize=11)
    ax2.set_ylabel(r"Pairs $m$", fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.savefig(filename, format='pdf', bbox_inches='tight')
    plt.show()


# Plots erzeugen
plot_lccf_with_pairs(
    lags_7mm, lccf_7mm, pairs_7mm, bin_width, 
    f"Cross-Correlation: MAGIC vs. Telamon 7 mm (Binning: {bin_width:.0f} d)", 
    "images/lccf_7mm_with_pairs.pdf"
)

plot_lccf_with_pairs(
    lags_14mm, lccf_14mm, pairs_14mm, bin_width, 
    f"Cross-Correlation: MAGIC vs. Telamon 14 mm (Binning: {bin_width:.0f} d)", 
    "images/lccf_14mm_with_pairs.pdf"
)
dt_median_gamma = np.median(np.diff(np.sort(t_gamma)))
dt_median_radio_7mm = np.median(np.diff(np.sort(t_radio)))
dt_median_radio_14mm = np.median(np.diff(np.sort(t_radio_14mm)))
print(f"Median Sampling: Gamma = {dt_median_gamma:.1f} d, Radio 7mm = {dt_median_radio_7mm:.1f} d, Radio 14mm = {dt_median_radio_14mm:.1f} d")

def report(lags, lccf, name, tau_max=300):
    ok = ~np.isnan(lccf)
    i_max, i_min = np.nanargmax(lccf), np.nanargmin(lccf)
    print(f"{name}: max r = {lccf[i_max]:.2f} at tau = {lags[i_max]:.1f} d, "
          f"min r = {lccf[i_min]:.2f} at tau = {lags[i_min]:.1f} d")
    sel = ok & (np.abs(lags) <= tau_max)
    j = np.nanargmax(np.where(sel, lccf, -np.inf))
    print(f"   |tau| <= {tau_max} d: max r = {lccf[j]:.2f} at tau = {lags[j]:.1f} d")

report(lags_7mm,  lccf_7mm,  "7 mm")
report(lags_14mm, lccf_14mm, "14 mm")

## Korrelationsheatmap

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.table import Table

# ---------------------------------------------------------------
# Configuration & Analysis Parameters
# ---------------------------------------------------------------
MIN_PAIRS = 10                       # Minimum number of valid pairs per window
LAG_MIN, LAG_MAX = -500, 500         # Lag search range in days
REF_BIN_WIDTH = 15.0                 # Fixed reference bin width for single plots [d]
WINDOW_WIDTHS = np.arange(5, 61, 1.0)          # Window widths for heatmap scan [d]
EVAL_LAGS = np.arange(LAG_MIN, LAG_MAX + 1, 1.0)    # Evaluation lags on a 1-day grid [d]
OUTPUT_DIR = "images"

# 1. Gamma-ray data (MAGIC)
flux_table_filtered = Table.read("table_lc_filtered.fits")
time_gamma = (flux_table_filtered["time_min"].data + flux_table_filtered["time_max"].data) / 2.0
flux_gamma = np.squeeze(flux_table_filtered["flux"].data)

# 2. Radio data (Effelsberg)
df_radio = pd.read_csv("Mrk501.csv")
radio_data_by_band = {}
for band in (7, 14):
    df_band = df_radio[df_radio["Band [mm]"] == band]
    radio_data_by_band[band] = {
        "time": df_band["MJD"].values,
        "flux": df_band["Flux Density [Jy]"].values,
    }


def lccf_single_window(delta_t_matrix, flux_1, flux_2, lag_low, lag_high, min_pairs):
    """
    Computes the LCCF for all pairs within the time lag interval [lag_low, lag_high).
    delta_t_matrix = t2 - t1 (Radio minus Gamma; positive lags indicate Radio lags Gamma).
    """
    # Identify indices of all pairs falling into the current lag window
    idx_flux2, idx_flux1 = np.where((delta_t_matrix >= lag_low) & (delta_t_matrix < lag_high))
    
    paired_flux1 = flux_1[idx_flux1]
    paired_flux2 = flux_2[idx_flux2]

    # Filter out missing/NaN values
    valid_mask = ~np.isnan(paired_flux1) & ~np.isnan(paired_flux2)
    paired_flux1 = paired_flux1[valid_mask]
    paired_flux2 = paired_flux2[valid_mask]
    
    n_valid_pairs = len(paired_flux1)
    if n_valid_pairs < min_pairs:
        return np.nan, n_valid_pairs

    std_flux1 = np.std(paired_flux1)
    std_flux2 = np.std(paired_flux2)
    
    # Avoid zero division
    if std_flux1 == 0 or std_flux2 == 0:
        return np.nan, n_valid_pairs

    mean_flux1 = np.mean(paired_flux1)
    mean_flux2 = np.mean(paired_flux2)
    
    lccf_value = np.mean((paired_flux1 - mean_flux1) * (paired_flux2 - mean_flux2)) / (std_flux1 * std_flux2)
    return lccf_value, n_valid_pairs


def lccf_fixed_bins(time_1, flux_1, time_2, flux_2, bin_width, lag_min=LAG_MIN, lag_max=LAG_MAX, min_pairs=MIN_PAIRS):
    """Computes the LCCF across fixed, adjacent bins (matching standard discrete plots)."""
    delta_t_matrix = time_2[:, None] - time_1[None, :]
    bin_edges = np.arange(lag_min, lag_max + bin_width, bin_width)
    lag_centers = (bin_edges[:-1] + bin_edges[1:]) / 2.0
    
    results = [
        lccf_single_window(delta_t_matrix, flux_1, flux_2, bin_edges[i], bin_edges[i + 1], min_pairs)
        for i in range(len(bin_edges) - 1)
    ]
    lccf_values = np.array([res[0] for res in results])
    pair_counts = np.array([res[1] for res in results])
    return lag_centers, lccf_values, pair_counts


def lccf_sliding_grid(time_1, flux_1, time_2, flux_2, window_widths, eval_lags, min_pairs=MIN_PAIRS):
    """Computes a 2D LCCF grid using sliding windows centered at each evaluation lag."""
    delta_t_matrix = time_2[:, None] - time_1[None, :]
    correlation_grid = np.full((len(eval_lags), len(window_widths)), np.nan)

    for j, width in enumerate(window_widths):
        half_width = width / 2.0
        for i, lag_center in enumerate(eval_lags):
            correlation_grid[i, j], _ = lccf_single_window(
                delta_t_matrix, flux_1, flux_2, lag_center - half_width, lag_center + half_width, min_pairs
            )
            
    return correlation_grid


# ---------------------------------------------------------------
# Heatmap Visualization
# ---------------------------------------------------------------
def make_heatmap(band_mm, time_radio, flux_radio, output_filename):
    # 1. Determine reference peak from fixed bins for tau > 0
    ref_lags, ref_lccf_values, _ = lccf_fixed_bins(
        time_gamma, flux_gamma, time_radio, flux_radio, REF_BIN_WIDTH
    )
    positive_lag_mask = ref_lags > 0
    idx_best_positive = np.nanargmax(np.where(positive_lag_mask, ref_lccf_values, -np.inf))
    
    ref_tau = ref_lags[idx_best_positive]
    ref_r = ref_lccf_values[idx_best_positive]
    print(
        f"{band_mm} mm: Highest value at tau > 0 (bin width {REF_BIN_WIDTH:.0f} d): "
        f"r = {ref_r:.2f} at tau = {ref_tau:.1f} d"
    )

    # 2. Compute 2D grid with sliding window
    correlation_grid = lccf_sliding_grid(
        time_gamma, flux_gamma, time_radio, flux_radio, WINDOW_WIDTHS, EVAL_LAGS
    )

    # 3. Plotting
    colormap = plt.get_cmap("coolwarm").copy()
    colormap.set_bad("white")  # White denotes NaN (insufficient pair counts)

    fig, ax = plt.subplots(figsize=(11, 7))
    mesh = ax.pcolormesh(
        WINDOW_WIDTHS, EVAL_LAGS, np.ma.masked_invalid(correlation_grid),
        cmap=colormap, vmin=-0.8, vmax=0.8, shading="auto"
    )
    
    cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
    cbar.set_label(r"LCCF coefficient $r$", fontsize=12)
    
    ax.axhline(0, color="gray", lw=1, ls="--", alpha=0.8)
    ax.axhline(
        ref_tau, color="black", lw=1.3, ls=":",
        label=rf"Highest $\tau > 0$ at $\Delta\tau = {REF_BIN_WIDTH:.0f}\,$d: $\tau = {ref_tau:.1f}\,$d, $r = {ref_r:.2f}$"
    )
    
    ax.set_xlabel(r"LCCF window width $\Delta\tau$ [d]", fontsize=12)
    ax.set_ylabel(r"Time lag $\tau$ [d] (Radio $-$ $\gamma$)", fontsize=12)
    ax.grid(True, alpha=0.2)
    ax.legend(loc="upper right", frameon=True)
    
    fig.tight_layout()
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    fig.savefig(os.path.join(OUTPUT_DIR, output_filename), format="pdf", bbox_inches="tight")
    plt.show()


# ---------------------------------------------------------------
# Execution
# ---------------------------------------------------------------
for band in (7, 14):
    time_r = radio_data_by_band[band]["time"]
    flux_r = radio_data_by_band[band]["flux"]
    make_heatmap(band, time_r, flux_r, f"lccf_{band}mm_binning_heatmap.pdf")

# !!! Change of Environments: Gammapy --> Sim. Lightcurves !!!

# Simulation der Lichtkurven nach Emmanoupoulos & Timmer (Code auf Basis der Dissertation von Lena Linhoff) 

   The light curve simulation and the PSD/PDF check plots in this notebook are adapted from
   `scripts/simulate_lightcurves.py` and `scripts/plots.py` of [lena-lin/emmanoulopoulos](https://github.com/lena-lin/emmanoulopoulos)
   (Copyright (c) 2021 lena-lin, MIT License, see `emmanoulopoulos/LICENSE.md`).
   Modified by Martin Bothen (2026).

In [ ]:
import os
import importlib
import json
import logging
from pathlib import Path
import sys

from astropy.table import Table
import astropy.units as u
import numpy as np
import pandas as pd
from tqdm import tqdm

# Add the (modified) emmanoulopoulos package to the path
repo_path = str(Path.cwd().parent / "emmanoulopoulos")
if repo_path not in sys.path:
    sys.path.append(repo_path)

import emmanoulopoulos.emmanoulopoulos_lc_simulation as sim_mod
import emmanoulopoulos.lightcurve as lc_mod

importlib.reload(lc_mod)
importlib.reload(sim_mod)

LC = lc_mod.LC
Emmanoulopoulos_Sampler = sim_mod.Emmanoulopoulos_Sampler

# Logger configuration
logger = logging.getLogger()
logger.setLevel(logging.INFO)
if logger.hasHandlers():
    logger.handlers.clear()
formatter = logging.Formatter(
    "%(asctime)s:%(levelname)s:%(message)s", datefmt="%Y-%m-%dT%H:%M:%S"
)
handler = logging.StreamHandler()
handler.setFormatter(formatter)
logger.addHandler(handler)


def run_lc_simulation(
    time,
    flux,
    err,
    name,
    output_path,
    n=10,
    scale_back_factor=1.0,
    tbin=3 * u.day,
):
    lc = LC(time, flux, err, tbin=tbin)
    lc.fit_PSD()
    pdf_parameter = lc.fit_PDF()

    # Refit until all PDF parameters are physically meaningful (non-negative)
    pdf_positive = False
    while not pdf_positive:
        for k, v in pdf_parameter.to_dict().items():
            if v < 0:
                pdf_parameter = lc.fit_PDF()
                break
        else:
            pdf_positive = True

    logger.info(f"[{name}] PDF parameters: {pdf_parameter.to_dict()}")
    logger.info(f"[{name}] PSD parameters: {lc.psd_parameter.to_dict()}")
    logger.info(f"Simulating {n} light curves for {name}...")

    # Emmanoulopoulos sampler without Poisson noise (fluxes, not counts)
    sampler = Emmanoulopoulos_Sampler(
        poisson_noise=False, tk_red_noise_factor=100, tk_alias_tbin=1
    )
    results = []
    for _ in tqdm(range(n), desc=f"Simulating {name}"):
        lc_sim = sampler.sample_from_lc(lc)
        results.append((np.array(lc_sim.original_flux) * scale_back_factor).tolist())

    simulated_lcs = {idx: res for idx, res in enumerate(results)}

    # Sanity check: all simulated light curves must be pairwise different
    X = np.array(results)
    n_unique = len(np.unique(X, axis=0))
    if n_unique < n:
        logger.error(f"Warning: only {n_unique} of {n} light curves are different!")
    else:
        logger.info("All light curves are pairwise different.")

    # Save the simulated light curves
    Path(output_path).mkdir(parents=True, exist_ok=True)
    json_file = Path(output_path, f"sim_lc_{name}_n{n}.json")
    with open(str(json_file), "w") as f:
        json.dump(
            simulated_lcs, f, indent=4, separators=(", ", ": "), sort_keys=True
        )
    logger.info(f"Saved to: {json_file}\n")


def load_data_MAGIC(filename):
    flux_table = Table.read(filename)
    mjd_mean = (flux_table["time_min"] + flux_table["time_max"]) / 2
    time = (mjd_mean - mjd_mean.min()) * u.day
    flux = flux_table["flux"].data.flatten()
    err = flux_table["flux_err"].data.flatten()
    time_gamma_mjd = mjd_mean
    return time_gamma_mjd, time, flux, err


if __name__ == "__main__":
    ########################################################################
    ############# CONFIGURATION ############################################
    ########################################################################
    output_path = "build"
    n_sims = 1000  # number of simulated light curves per series
    source_name = "MRK_501"
    t_bin_MAGIC = 3 * u.day  # simulation grid for the gamma-ray light curve
    t_bin_Effel = 10 * u.day  # simulation grid for the radio light curves
    filename_magic = "table_lc_filtered.fits"  # night-wise MAGIC light curve (upper-limit nights removed)
    path_Radio = "Mrk501.csv"  # Effelsberg/TELAMON flux densities

    # ------------------------------------------
    # 1. MAGIC data
    # ------------------------------------------
    time_gamma_mjd, time_g, flux_g, err_g = load_data_MAGIC(filename_magic)
    duration_magic = time_g.max()
    print(
        f"MAGIC time span: {duration_magic:.2f} (approx. {int(duration_magic.value)} days)"
    )

    # Normalize to the mean flux before the fits; simulated curves are scaled back
    mean_flux_magic = np.mean(flux_g)
    scale_factor_magic = 1.0 / mean_flux_magic
    flux_g_scaled = flux_g * scale_factor_magic
    err_g_scaled = err_g * scale_factor_magic

    run_lc_simulation(
        time_g,
        flux_g_scaled,
        err_g_scaled,
        name=f"{source_name}_MAGIC",
        n=n_sims,
        output_path=output_path,
        scale_back_factor=mean_flux_magic,
        tbin=t_bin_MAGIC,
    )

    # ------------------------------------------
    # 2. Radio data
    # ------------------------------------------
    df_radio = pd.read_csv(path_Radio)

    # 2a. Radio 7 mm Band
    df_7mm = df_radio[df_radio["Band [mm]"] == 7].copy()
    if not df_7mm.empty:
        df_7mm.sort_values(by="MJD", ascending=True, inplace=True)
        t_radio_7mm_mjd = df_7mm["MJD"].values
        time_radio_7mm = (t_radio_7mm_mjd - t_radio_7mm_mjd.min()) * u.day
        duration_radio_7mm = time_radio_7mm.max()
        print(
            f"Radio 7 mm time span: {duration_radio_7mm:.2f} (approx. {int(duration_radio_7mm.value)} days)"
        )

        f_radio_7mm = df_7mm["Flux Density [Jy]"].values
        e_radio_7mm = df_7mm["Error [Jy]"].values

        mean_flux_7mm = np.mean(f_radio_7mm)
        scale_factor_7mm = 1.0 / mean_flux_7mm
        f_radio_7mm_scaled = f_radio_7mm * scale_factor_7mm
        e_radio_7mm_scaled = e_radio_7mm * scale_factor_7mm

        run_lc_simulation(
            time_radio_7mm,
            f_radio_7mm_scaled,
            e_radio_7mm_scaled,
            name=f"{source_name}_RADIO_7mm",
            n=n_sims,
            output_path=output_path,
            scale_back_factor=mean_flux_7mm,
            tbin=t_bin_Effel,
        )
    else:
        logger.warning(f"No 7 mm data found in {path_Radio}!")

    # 2b. Radio 14 mm Band
    df_14mm = df_radio[df_radio["Band [mm]"] == 14].copy()
    if not df_14mm.empty:
        df_14mm.sort_values(by="MJD", ascending=True, inplace=True)
        t_radio_14mm_mjd = df_14mm["MJD"].values
        time_radio_14mm = (t_radio_14mm_mjd - t_radio_14mm_mjd.min()) * u.day
        duration_radio_14mm = time_radio_14mm.max()
        print(
            f"Radio 14 mm time span: {duration_radio_14mm:.2f} (approx. {int(duration_radio_14mm.value)} days)"
        )

        f_radio_14mm = df_14mm["Flux Density [Jy]"].values
        e_radio_14mm = df_14mm["Error [Jy]"].values

        mean_flux_14mm = np.mean(f_radio_14mm)
        scale_factor_14mm = 1.0 / mean_flux_14mm
        f_radio_14mm_scaled = f_radio_14mm * scale_factor_14mm
        e_radio_14mm_scaled = e_radio_14mm * scale_factor_14mm

        run_lc_simulation(
            time_radio_14mm,
            f_radio_14mm_scaled,
            e_radio_14mm_scaled,
            name=f"{source_name}_RADIO_14mm",
            n=n_sims,
            output_path=output_path,
            scale_back_factor=mean_flux_14mm,
            tbin=t_bin_Effel,
        )
    else:
        logger.warning(f"No 14 mm data found in {path_Radio}!")

## Generien PDF & PSD Plots Gamma

In [ ]:
import os
import numpy as np
import astropy.units as u
from emmanoulopoulos.emmanoulopoulos_lc_simulation import Emmanoulopoulos_Sampler
from emmanoulopoulos.models import power_spectral_density, pdf_gamma_lognorm
import matplotlib.pyplot as plt

def plot_emma_gamma_separate():
    N =  632
    tbin = 3 * u.day
    

    pdf_params = {
    'a': 1.7471687495337178,
    's': 0.3192836973325601,
    'loc': 0.0,
    'scale': 0.7501225282908724,
    'p': 0.15107002811928164
    }

    psd_params = {
    'A': 0.062379969135851025,
    'alpha_low': 1.0000007091187026,
    'alpha_high': 3.0866917635669466,
    'f_bend': 0.020094569253762323,
    'c': 0.019412462782384354
    }

    sampler = Emmanoulopoulos_Sampler(poisson_noise=False, tk_red_noise_factor=100, tk_alias_tbin=1)
    print("Simulating MAGIC light curve...")

    sim_lc = sampler.sample_from_psd_pdf(psd_params, pdf_params, N, tbin)

    print("Calculating models...")

    psd_fit_sim_lc = sim_lc.fit_PSD(window=True).to_dict()
    pdf_fit_sim_lc = sim_lc.fit_PDF(unbinned=False).to_dict()

    os.makedirs("build", exist_ok=True)

    # ==========================================
    # PLOT 1: PSD (Power Spectral Density)
    # ==========================================
    plt.figure(figsize=(6.5, 4.5), constrained_layout=True) 
    
    f = sim_lc._f_periodogram[1:]
    p = sim_lc._periodogram[1:]

    plt.plot(f, p, 'o', alpha=0.3, color='#1f77b4', markersize=5, label='Simulation (Data points)')
    plt.plot(f, power_spectral_density(f, **psd_params), color='#ff7f0e', linewidth=2.5, label='Model (MAGIC Original)')
    plt.plot(f, power_spectral_density(f, **psd_fit_sim_lc), color='#2ca02c', linewidth=2, linestyle='--', label='Simulation (Fit)')
    plt.xscale('log')
    plt.yscale('log')
    plt.grid(True, which="both", linestyle="--", alpha=0.4)
    
    plt.title('Power Spectral Density (PSD) - MAGIC', fontsize=14, fontweight='bold')
    plt.xlabel(r'Frequency $\nu$ [d$^{-1}$]', fontsize=12)
    plt.ylabel(r'Power [$(\Phi/\langle\Phi\rangle)^2 \cdot \mathrm{d}$]', fontsize=12)
    plt.legend(fontsize=10, loc='lower left')

    plt.savefig("build/emma_magic_psd.pdf", dpi=300)
    plt.show()
    plt.close() 
    print("-> Done! PSD saved as 'build/emma_magic_psd.pdf'")

    # ==========================================
    # PLOT 2: PDF (Probability Density Function)
    # ==========================================
    plt.figure(figsize=(6.5, 4.5), constrained_layout=True)
    
    hist, edges, _ = plt.hist(sim_lc.original_flux, bins='auto', density=True, alpha=0.4, color='#1f77b4', edgecolor='white', label='Simulation (Histogram)')
    c = (edges[:-1] + edges[1:]) / 2
    plt.plot(c, pdf_gamma_lognorm(c, **pdf_params), color='#ff7f0e', linewidth=2.5, label="Model (MAGIC Original)")
    plt.plot(c, pdf_gamma_lognorm(c, **pdf_fit_sim_lc), color='#2ca02c', linewidth=2, linestyle='--', label="Simulation (Fit)")
    
    plt.grid(True, linestyle="--", alpha=0.4)
    
    plt.xlabel(r'Scaled Flux $\Phi / \langle\Phi\rangle$', fontsize=12)
    plt.ylabel('Probability Density', fontsize=12)
    plt.legend(fontsize=10, loc='upper right')
    
    plt.savefig("build/emma_magic_pdf.pdf", dpi=300)
    plt.show()
    plt.close() 
    print("-> Done! PDF saved as 'build/emma_magic_pdf.pdf'")


if __name__ == "__main__":
    plot_emma_gamma_separate()

## Generieren PSD PDF Plots Radio 

In [ ]:
import os
import numpy as np
import astropy.units as u
from emmanoulopoulos.emmanoulopoulos_lc_simulation import Emmanoulopoulos_Sampler
from emmanoulopoulos.models import power_spectral_density, pdf_gamma_lognorm
import matplotlib.pyplot as plt

def plot_emma_radio_separate():
    N = 188
    tbin = 10 * u.day

    pdf_params = {
    'a': 1.0578789550794447,
    's': 0.08819834781598414,
    'loc': 4.713075658868812e-05,
    'scale': 1.0049020862285234,
    'p': 0.17237602521238174
    }

    psd_params = {
    'A': 2.9337565782018072e-05,
    'alpha_low': 1.7359450551255975,
    'alpha_high': 4.557034006318662,
    'f_bend': 0.016456449694585006,
    'c': 3.208375621384027e-10
    }
    
    sampler = Emmanoulopoulos_Sampler(poisson_noise=False, tk_red_noise_factor=100, tk_alias_tbin=1)
    print("Simulating Radio light curve...")

    sim_lc = sampler.sample_from_psd_pdf(psd_params, pdf_params, N, tbin)

    print("Calculating models...")

    psd_fit_sim_lc = sim_lc.fit_PSD(window=True).to_dict()
    pdf_fit_sim_lc = sim_lc.fit_PDF(unbinned=False).to_dict()

    os.makedirs("build", exist_ok=True)

    # ==========================================
    # PLOT 1: PSD (Power Spectral Density)
    # ==========================================
    plt.figure(figsize=(6.5, 4.5), constrained_layout=True) 

    f = sim_lc._f_periodogram[1:]
    p = sim_lc._periodogram[1:]

    plt.plot(f, p, 'o', alpha=0.3, color='#1f77b4', markersize=5, label='Simulation (Data points)')
    
    # Hier nutzen nun psd_plot
    plt.plot(f, power_spectral_density(f, **psd_params), color='#ff7f0e', linewidth=2.5, label='Model (Effelsberg Original)')
    plt.plot(f, power_spectral_density(f, **psd_fit_sim_lc), color='#2ca02c', linewidth=2, linestyle='--', label='Simulation (Fit)')
    
    plt.xscale('log')
    plt.yscale('log')
    plt.grid(True, which="both", linestyle="--", alpha=0.4)
    
    plt.title('Power Spectral Density (PSD) - Effelsberg', fontsize=14, fontweight='bold')
    plt.xlabel(r'Frequency $\nu$ [d$^{-1}$]', fontsize=12)
    plt.ylabel(r'Power [$(\Phi/\langle\Phi\rangle)^2 \cdot \mathrm{d}$]', fontsize=12)
    plt.legend(fontsize=10, loc='lower left')

    plt.savefig("build/emma_effelsberg_psd_7mm.pdf", dpi=300)
    plt.show()
    plt.close() 
    print("-> Done! PSD saved as 'build/emma_effelsberg_psd.pdf'")

    # ==========================================
    # PLOT 2: PDF (Probability Density Function)
    # ==========================================
    plt.figure(figsize=(6.5, 4.5), constrained_layout=True)
    
    hist, edges, _ = plt.hist(sim_lc.original_flux, bins='auto', density=True, alpha=0.4, color='#1f77b4', edgecolor='white', label='Simulation (Histogram)')
    c = (edges[:-1] + edges[1:]) / 2
    plt.plot(c, pdf_gamma_lognorm(c, **pdf_params), color='#ff7f0e', linewidth=2.5, label="Model (Effelsberg Original)")
    plt.plot(c, pdf_gamma_lognorm(c, **pdf_fit_sim_lc), color='#2ca02c', linewidth=2, linestyle='--', label="Simulation (Fit)")
    
    plt.grid(True, linestyle="--", alpha=0.4)
    
    plt.xlabel(r'Scaled Flux $\Phi / \langle\Phi\rangle$', fontsize=12)
    plt.ylabel('Probability Density', fontsize=12)
    plt.legend(fontsize=10, loc='upper right')
    
    plt.savefig("build/emma_effelsberg_pdf_7mm.pdf", dpi=300)
    plt.show()
    plt.close() 
    print("-> Done! PDF saved as 'build/emma_effelsberg_pdf.pdf'")


if __name__ == "__main__":
    plot_emma_radio_separate()

In [ ]:
import os
import numpy as np
import astropy.units as u
from emmanoulopoulos.emmanoulopoulos_lc_simulation import Emmanoulopoulos_Sampler
from emmanoulopoulos.models import power_spectral_density, pdf_gamma_lognorm
import matplotlib.pyplot as plt

def plot_emma_radio_separate():
    N = 191
    tbin = 10 * u.day


    pdf_params = {
    'a': 1.3816443851674725,
    's': 0.10221105338891207,
    'loc': 0.0,
    'scale': 0.9668663552242258,
    'p': 6.498279602677563e-11
    }

    psd_params = {
    'A': 0.00031586356696907075,
    'alpha_low': 1.064364265732928,
    'alpha_high': 3.7968542517007027,
    'f_bend': 0.007516139215433459,
    'c': 9.055696851845987e-05
    }
    
    
    sampler = Emmanoulopoulos_Sampler(poisson_noise=False, tk_red_noise_factor=100, tk_alias_tbin=1)
    print("Simulating Radio light curve...")

    sim_lc = sampler.sample_from_psd_pdf(psd_params, pdf_params, N, tbin)

    print("Calculating models...")

    psd_fit_sim_lc = sim_lc.fit_PSD(window=True).to_dict()
    pdf_fit_sim_lc = sim_lc.fit_PDF(unbinned=False).to_dict()

    os.makedirs("build", exist_ok=True)

    # ==========================================
    # PLOT 1: PSD (Power Spectral Density)
    # ==========================================
    plt.figure(figsize=(6.5, 4.5), constrained_layout=True) 

    f = sim_lc._f_periodogram[1:]
    p = sim_lc._periodogram[1:]


    plt.plot(f, p, 'o', alpha=0.3, color='#1f77b4', markersize=5, label='Simulation (Data points)')
    
    # Hier nutzen nun psd_plot
    plt.plot(f, power_spectral_density(f, **psd_params), color='#ff7f0e', linewidth=2.5, label='Model (Effelsberg Original)')
    plt.plot(f, power_spectral_density(f, **psd_fit_sim_lc), color='#2ca02c', linewidth=2, linestyle='--', label='Simulation (Fit)')
    
    plt.xscale('log')
    plt.yscale('log')
    plt.grid(True, which="both", linestyle="--", alpha=0.4)
    
    plt.xlabel(r'Frequency $\nu$ [d$^{-1}$]', fontsize=12)
    plt.ylabel(r'Power [$(\Phi/\langle\Phi\rangle)^2 \cdot \mathrm{d}$]', fontsize=12)
    plt.legend(fontsize=10, loc='lower left')

    plt.savefig("build/emma_effelsberg_psd_14mm.pdf", dpi=300)
    plt.show()
    plt.close() 
    print("-> Done! PSD saved as 'build/emma_effelsberg_psd.pdf'")

    # ==========================================
    # PLOT 2: PDF (Probability Density Function)
    # ==========================================
    plt.figure(figsize=(6.5, 4.5), constrained_layout=True)
    
    hist, edges, _ = plt.hist(sim_lc.original_flux, bins='auto', density=True, alpha=0.4, color='#1f77b4', edgecolor='white', label='Simulation (Histogram)')
    c = (edges[:-1] + edges[1:]) / 2
    plt.plot(c, pdf_gamma_lognorm(c, **pdf_params), color='#ff7f0e', linewidth=2.5, label="Model (Effelsberg Original)")
    plt.plot(c, pdf_gamma_lognorm(c, **pdf_fit_sim_lc), color='#2ca02c', linewidth=2, linestyle='--', label="Simulation (Fit)")
    
    plt.grid(True, linestyle="--", alpha=0.4)
    
    plt.title('Probability Density Function (PDF) - Effelsberg', fontsize=14, fontweight='bold')
    plt.xlabel(r'Scaled Flux $\Phi / \langle\Phi\rangle$', fontsize=12)
    plt.ylabel('Probability Density', fontsize=12)
    plt.legend(fontsize=10, loc='upper right')
    
    plt.savefig("build/emma_effelsberg_pdf_14mm.pdf", dpi=300)
    plt.show()
    plt.close() 
    print("-> Done! PDF saved as 'build/emma_effelsberg_pdf.pdf'")


if __name__ == "__main__":
    plot_emma_radio_separate()

# Signifikanzen

# 7mm Band

In [ ]:
# Significance bands of the LCCF from simulated light curves
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

# Gamma-ray data
from astropy.table import Table

flux_table = Table.read("table_lc_filtered.fits")   # night-wise MAGIC light curve of Mrk 501
t_gamma = np.asarray((flux_table["time_min"] + flux_table["time_max"]) / 2.0, dtype=float)
y_gamma = np.asarray(flux_table["flux"].quantity.value.flatten(), dtype=float)


# Radio data (7 mm, sorted by time)
df_radio = pd.read_csv("Mrk501.csv")
df_7mm = df_radio[df_radio["Band [mm]"] == 7].copy()
df_7mm.sort_values(by="MJD", ascending=True, inplace=True)

t_radio = df_7mm["MJD"].values
f_radio = df_7mm["Flux Density [Jy]"].values

t_gamma_sim = t_gamma
t_radio_sim = t_radio

BIN_W = 15  # lag bin width [d]
N_MIN = 10  # minimum number of pairs per lag bin
bins = np.arange(-500, 500 + BIN_W, BIN_W)
lags = (bins[:-1] + bins[1:]) / 2.0

# Pair indices per lag bin (identical for all simulated curves, since the sampling is the same)
print("Precomputing lag bin indices...")
dt = t_radio_sim[:, None] - t_gamma_sim[None, :]
bin_pair_indices = []

for i in range(len(bins) - 1):
    bin_mask = (dt >= bins[i]) & (dt < bins[i + 1])
    if np.sum(bin_mask) > N_MIN:
        idx_r, idx_g = np.where(bin_mask)
        bin_pair_indices.append((idx_g, idx_r))
    else:
        bin_pair_indices.append(None)

print("Loading simulated light curves...")
with open("build/sim_lc_MRK_501_MAGIC_n1000.json", "r") as f:
    sim_gamma_data = json.load(f)
with open("build/sim_lc_MRK_501_RADIO_7mm_n1000.json", "r") as f:
    sim_radio_data = json.load(f)

n_gamma = len(sim_gamma_data)  # 1000
n_radio = len(sim_radio_data)  # 1000

# Convert to 2D arrays of shape (n_sim, n_epochs)
Y_gamma = np.array([sim_gamma_data[str(i)] for i in range(n_gamma)], dtype=np.float32)
Y_radio = np.array([sim_radio_data[str(i)] for i in range(n_radio)], dtype=np.float32)

# Mask unphysical values <= 0 as NaN
Y_gamma[Y_gamma <= 0] = np.nan
Y_radio[Y_radio <= 0] = np.nan

def correlate_all(Y_gamma, Y_radio):
    """
    Compute the LCCF of every simulated gamma-ray curve with every
    simulated radio curve.
    Returns an array of shape (n_gamma * n_radio, n_lags)
    """
    n_bins = len(bin_pair_indices)
    results = []

    for y_g in tqdm(Y_gamma, desc="Simulated LCCFs"):
        for y_r in Y_radio:
            lccf_vals = np.empty(n_bins, dtype=np.float32)
            lccf_vals[:] = np.nan

            for b_idx, pair in enumerate(bin_pair_indices):
                if pair is None:
                    continue

                idx_g, idx_r = pair
                a_i = y_g[idx_g]
                b_j = y_r[idx_r]

                # Use only valid (non-NaN) pairs
                valid = ~np.isnan(a_i) & ~np.isnan(b_j)
                m_valid = np.sum(valid)

                if m_valid > N_MIN:
                    a_valid = a_i[valid]
                    b_valid = b_j[valid]

                    mu_a, sig_a = np.mean(a_valid), np.std(a_valid)
                    mu_b, sig_b = np.mean(b_valid), np.std(b_valid)

                    if sig_a > 0 and sig_b > 0:
                        lccf_vals[b_idx] = (1.0 / m_valid) * np.sum(
                            ((a_valid - mu_a) * (b_valid - mu_b)) / (sig_a * sig_b)
                        )

            results.append(lccf_vals)

    return np.array(results, dtype=np.float32)

all_mock_lccfs = correlate_all(Y_gamma, Y_radio)

# 2 sigma (95.45 %): 2.275 % to 97.725 % percentiles per lag bin
low_95 = np.nanpercentile(all_mock_lccfs, 2.275, axis=0)
high_95 = np.nanpercentile(all_mock_lccfs, 97.725, axis=0)

# 3 sigma (99.73 %): 0.135 % to 99.865 % percentiles per lag bin
low_99 = np.nanpercentile(all_mock_lccfs, 0.135, axis=0)
high_99 = np.nanpercentile(all_mock_lccfs, 99.865, axis=0)

# LCCF of the observed light curves with the same settings
def calculate_lccf_real(t1, y1, t2, y2, lag_bins):
    dt_local = t2[:, None] - t1[None, :]
    lags_local = (lag_bins[:-1] + lag_bins[1:]) / 2.0
    vals = []

    for i in range(len(lag_bins) - 1):
        mask = (dt_local >= lag_bins[i]) & (dt_local < lag_bins[i + 1])
        m = np.sum(mask)
        if m > N_MIN:
            idx2, idx1 = np.where(mask)
            a_i = y1[idx1]
            b_j = y2[idx2]
            valid = ~np.isnan(a_i) & ~np.isnan(b_j)
            if np.sum(valid) > N_MIN:
                a_v = a_i[valid]
                b_v = b_j[valid]
                mu_a, s_a = np.mean(a_v), np.std(a_v)
                mu_b, s_b = np.mean(b_v), np.std(b_v)
                if s_a > 0 and s_b > 0:
                    val = (1.0 / len(a_v)) * np.sum(((a_v - mu_a) * (b_v - mu_b)) / (s_a * s_b))
                    vals.append(val)
                else:
                    vals.append(np.nan)
            else:
                vals.append(np.nan)
        else:
            vals.append(np.nan)
    return lags_local, np.array(vals)

lags_real, lccf_real = calculate_lccf_real(t_gamma, y_gamma, t_radio, f_radio, bins)

plt.figure(figsize=(11, 6))

# Measured LCCF
plt.scatter(lags_real, lccf_real, color="black", s=25, zorder=5, label="Measured LCCF (Mrk 501)")
#plt.plot(lags_real, lccf_real, color="black", alpha=0.4, lw=1, zorder=4)

# 3 sigma limits
plt.plot(lags, high_99, color="crimson", linestyle="--", linewidth=1.5, label=r"$3\sigma$ significance limit ($N=10^6$)", zorder=3)
plt.plot(lags, low_99, color="crimson", linestyle="--", linewidth=1.5, zorder=3)

# 2 sigma limits
plt.plot(lags, high_95, color="darkorange", linestyle=":", linewidth=2, label=r"$2\sigma$ significance limit ($N=10^6$)", zorder=3)
plt.plot(lags, low_95, color="darkorange", linestyle=":", linewidth=2, zorder=3)

# Layout
plt.axhline(0, color="gray", lw=1, ls="--")
plt.axvline(0, color="lightgray", lw=0.8, ls=":")
plt.xlabel(r"Time Lag $\tau$ [Days] (Radio $-$ $\gamma$)", fontsize=12)
plt.ylabel("LCCF Value", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.savefig("images/lccf_significance_mrk501_7mm.pdf", bbox_inches="tight")
plt.show()

# 14mm Band

In [ ]:
# Significance bands of the LCCF from simulated light curves
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

# Gamma-ray data
from astropy.table import Table

flux_table = Table.read("table_lc_filtered.fits")   # night-wise MAGIC light curve of Mrk 501
t_gamma = np.asarray((flux_table["time_min"] + flux_table["time_max"]) / 2.0, dtype=float)
y_gamma = np.asarray(flux_table["flux"].quantity.value.flatten(), dtype=float)


# Radio data (7 mm, sorted by time)
df_radio = pd.read_csv("Mrk501.csv")
df_14mm = df_radio[df_radio["Band [mm]"] == 14].copy()
df_14mm.sort_values(by="MJD", ascending=True, inplace=True)

t_radio = df_14mm["MJD"].values
f_radio = df_14mm["Flux Density [Jy]"].values

t_gamma_sim = t_gamma
t_radio_sim = t_radio

BIN_W = 15  # lag bin width [d]
N_MIN = 10  # minimum number of pairs per lag bin
bins = np.arange(-500, 500 + BIN_W, BIN_W)
lags = (bins[:-1] + bins[1:]) / 2.0

# Pair indices per lag bin (identical for all simulated curves, since the sampling is the same)
print("Precomputing lag bin indices...")
dt = t_radio_sim[:, None] - t_gamma_sim[None, :]
bin_pair_indices = []

for i in range(len(bins) - 1):
    bin_mask = (dt >= bins[i]) & (dt < bins[i + 1])
    if np.sum(bin_mask) > N_MIN:
        idx_r, idx_g = np.where(bin_mask)
        bin_pair_indices.append((idx_g, idx_r))
    else:
        bin_pair_indices.append(None)

print("Loading simulated light curves...")
with open("build/sim_lc_MRK_501_MAGIC_n1000.json", "r") as f:
    sim_gamma_data = json.load(f)
with open("build/sim_lc_MRK_501_RADIO_14mm_n1000.json", "r") as f:
    sim_radio_data = json.load(f)

n_gamma = len(sim_gamma_data)  # 1000
n_radio = len(sim_radio_data)  # 1000

# Convert to 2D arrays of shape (n_sim, n_epochs)
Y_gamma = np.array([sim_gamma_data[str(i)] for i in range(n_gamma)], dtype=np.float32)
Y_radio = np.array([sim_radio_data[str(i)] for i in range(n_radio)], dtype=np.float32)

# Mask unphysical values <= 0 as NaN
Y_gamma[Y_gamma <= 0] = np.nan
Y_radio[Y_radio <= 0] = np.nan

def correlate_all(Y_gamma, Y_radio):
    """
    Compute the LCCF of every simulated gamma-ray curve with every
    simulated radio curve.
    Returns an array of shape (n_gamma * n_radio, n_lags)
    """
    n_bins = len(bin_pair_indices)
    results = []

    for y_g in tqdm(Y_gamma, desc="Simulated LCCFs"):
        for y_r in Y_radio:
            lccf_vals = np.empty(n_bins, dtype=np.float32)
            lccf_vals[:] = np.nan

            for b_idx, pair in enumerate(bin_pair_indices):
                if pair is None:
                    continue

                idx_g, idx_r = pair
                a_i = y_g[idx_g]
                b_j = y_r[idx_r]

                # Use only valid (non-NaN) pairs
                valid = ~np.isnan(a_i) & ~np.isnan(b_j)
                m_valid = np.sum(valid)

                if m_valid > N_MIN:
                    a_valid = a_i[valid]
                    b_valid = b_j[valid]

                    mu_a, sig_a = np.mean(a_valid), np.std(a_valid)
                    mu_b, sig_b = np.mean(b_valid), np.std(b_valid)

                    if sig_a > 0 and sig_b > 0:
                        lccf_vals[b_idx] = (1.0 / m_valid) * np.sum(
                            ((a_valid - mu_a) * (b_valid - mu_b)) / (sig_a * sig_b)
                        )

            results.append(lccf_vals)

    return np.array(results, dtype=np.float32)

all_mock_lccfs = correlate_all(Y_gamma, Y_radio)

# 2 sigma (95.45 %): 2.275 % to 97.725 % percentiles per lag bin
low_95 = np.nanpercentile(all_mock_lccfs, 2.275, axis=0)
high_95 = np.nanpercentile(all_mock_lccfs, 97.725, axis=0)

# 3 sigma (99.73 %): 0.135 % to 99.865 % percentiles per lag bin
low_99 = np.nanpercentile(all_mock_lccfs, 0.135, axis=0)
high_99 = np.nanpercentile(all_mock_lccfs, 99.865, axis=0)

# LCCF of the observed light curves with the same settings
def calculate_lccf_real(t1, y1, t2, y2, lag_bins):
    dt_local = t2[:, None] - t1[None, :]
    lags_local = (lag_bins[:-1] + lag_bins[1:]) / 2.0
    vals = []

    for i in range(len(lag_bins) - 1):
        mask = (dt_local >= lag_bins[i]) & (dt_local < lag_bins[i + 1])
        m = np.sum(mask)
        if m > N_MIN:
            idx2, idx1 = np.where(mask)
            a_i = y1[idx1]
            b_j = y2[idx2]
            valid = ~np.isnan(a_i) & ~np.isnan(b_j)
            if np.sum(valid) > N_MIN:
                a_v = a_i[valid]
                b_v = b_j[valid]
                mu_a, s_a = np.mean(a_v), np.std(a_v)
                mu_b, s_b = np.mean(b_v), np.std(b_v)
                if s_a > 0 and s_b > 0:
                    val = (1.0 / len(a_v)) * np.sum(((a_v - mu_a) * (b_v - mu_b)) / (s_a * s_b))
                    vals.append(val)
                else:
                    vals.append(np.nan)
            else:
                vals.append(np.nan)
        else:
            vals.append(np.nan)
    return lags_local, np.array(vals)

lags_real, lccf_real = calculate_lccf_real(t_gamma, y_gamma, t_radio, f_radio, bins)

plt.figure(figsize=(11, 6))

# Measured LCCF
plt.scatter(lags_real, lccf_real, color="black", s=25, zorder=5, label="Measured LCCF (Mrk 501)")
#plt.plot(lags_real, lccf_real, color="black", alpha=0.4, lw=1, zorder=4)

# 3 sigma limits
plt.plot(lags, high_99, color="crimson", linestyle="--", linewidth=1.5, label=r"$3\sigma$ significance limit ($N=10^6$)", zorder=3)
plt.plot(lags, low_99, color="crimson", linestyle="--", linewidth=1.5, zorder=3)

# 2 sigma limits
plt.plot(lags, high_95, color="darkorange", linestyle=":", linewidth=2, label=r"$2\sigma$ significance limit ($N=10^6$)", zorder=3)
plt.plot(lags, low_95, color="darkorange", linestyle=":", linewidth=2, zorder=3)

# Layout
plt.axhline(0, color="gray", lw=1, ls="--")
plt.axvline(0, color="lightgray", lw=0.8, ls=":")
plt.xlabel(r"Time Lag $\tau$ [Days] (Radio $-$ $\gamma$)", fontsize=12)
plt.ylabel("LCCF Value", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.savefig("images/lccf_significance_mrk501_14mm.pdf", bbox_inches="tight")
plt.show()